# 07 - Modelo principal: CNN 1D + BiGRU + CTC

**Responsable:** Leonardo Mejía

Modelo principal según la selección de la revisión bibliográfica (Avance 2). Usa la base común de
`06_preparacion_modelado` (mismos datos, mismo split por participante y mismo loop de entrenamiento),
así que sus resultados se pueden comparar directamente con los demás modelos.

In [1]:
import sys
sys.path.append("..")

import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from src import dataset, features, metrics, train, vocab
from src.models import cnn_bigru

sns.set_theme(style="whitegrid")
FIGS = "../reports/figures"
train.fijar_semilla(42)

## 1. Por qué este modelo

Cada parte del modelo responde a un hallazgo del EDA o a la revisión bibliográfica:

| Hallazgo | Decisión | Fuente |
|----------|----------|--------|
| Secuencias con mediana de 147 cuadros y cola hasta ~750 | CNN 1D con *stride* que acorta la secuencia antes de la parte recurrente | Bai et al. (2018); Korotaev et al. (2025) |
| Las letras se distinguen por movimientos de pocos cuadros | Filtros de 5 cuadros sobre el eje del tiempo | LeCun et al. (1998) |
| Coarticulación entre letras y letras parecidas (M, N, T) | GRU **bidireccional**: cada letra se decide con contexto de antes y después; la tarea es *offline* | Schuster y Paliwal (1997); Graves y Schmidhuber (2005) |
| Solo 1,315 secuencias de entrenamiento y cómputo en CPU | GRU en vez de LSTM: rendimiento comparable con menos parámetros | Cho et al. (2014); Chung et al. (2014) |
| La etiqueta es la frase completa, sin alineación cuadro-letra | Pérdida CTC | Graves et al. (2006) |
| Mano derecha ausente en ~54.5 % de los cuadros | Máscara de detección como entrada (ya incluida en las 227 features) | Avance 1 |

La misma estructura (extractor convolucional + BiGRU + CTC) es la de HandReader, que con *landmarks* logró
69.3 % de exactitud por letra en ChicagoFSWild (Korotaev et al., 2025).

## 2. Arquitectura

```
x (B, T, 227) ─► Conv1d k=5, stride s ─► Conv1d k=3 ─► BiGRU (2 capas) ─► Linear ─► log_softmax (B, T/s, 60)
```

Cada bloque convolucional lleva BatchNorm, GELU y Dropout. La BiGRU recibe las secuencias empaquetadas
(`pack_padded_sequence`) para que la dirección hacia atrás no lea el relleno de las secuencias cortas.

In [2]:
loaders = dataset.crear_dataloaders(batch_size=32, max_frames=384)
x, x_len, y, y_len = next(iter(loaders["val"]))

modelo = cnn_bigru.construir()
log_probs, out_len = modelo(x, x_len)
print(modelo)
print()
print("entrada:", tuple(x.shape), "-> salida:", tuple(log_probs.shape))
print("largos de entrada:", x_len[:6].tolist())
print("largos de salida: ", out_len[:6].tolist())
print("parametros:", f"{train.n_parametros(modelo):,}")

CNNBiGRU(
  (convs): ModuleList(
    (0): BloqueConv(
      (conv): Conv1d(227, 192, kernel_size=(5,), stride=(2,), padding=(2,))
      (norm): BatchNorm1d(192, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (act): GELU(approximate='none')
      (drop): Dropout(p=0.3, inplace=False)
    )
    (1): BloqueConv(
      (conv): Conv1d(192, 192, kernel_size=(3,), stride=(1,), padding=(1,))
      (norm): BatchNorm1d(192, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (act): GELU(approximate='none')
      (drop): Dropout(p=0.3, inplace=False)
    )
  )
  (gru): GRU(192, 128, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (drop): Dropout(p=0.3, inplace=False)
  (salida): Linear(in_features=256, out_features=60, bias=True)
)

entrada: (32, 384, 227) -> salida: (32, 192, 60)
largos de entrada: [127, 343, 91, 173, 70, 167]
largos de salida:  [64, 172, 46, 87, 35, 84]
parametros: 888,828


## 3. ¿Cuánto se puede reducir el tiempo?

CTC solo puede alinear si la salida tiene al menos tantos cuadros como letras tiene la frase (más uno por
cada letra repetida seguida, como la doble *l* de *hello*). Se revisa en los datos de entrenamiento qué
*stride* cumple esa condición.

In [2]:
cache = dataset.cargar_cache()
idx_train = dataset.indices_de(cache, "train")
largos = np.minimum(np.diff(cache["offsets"])[idx_train], 384)
frases = cache["phrase"][idx_train]
minimo_ctc = np.array([len(f) + sum(a == b for a, b in zip(f, f[1:])) for f in frases])

filas = []
for stride in (1, 2, 3, 4, 6):
    salida = (largos - 1) // stride + 1
    filas.append({"stride": stride, "cuadros_mediana": int(np.median(salida)),
                  "pct_secuencias_alineables": round(100 * (salida >= minimo_ctc).mean(), 1)})
pd.DataFrame(filas)

,stride,cuadros_mediana,pct_secuencias_alineables
0,1,148,96.4
1,2,74,95.5
2,3,50,94.4
3,4,37,92.5
4,6,25,84.1


Incluso sin reducir el tiempo (*stride* 1), 3.6 % de las secuencias son más cortas que su frase: el
deletreo fue tan rápido que no hay suficientes cuadros. Con *stride* 2 la pérdida es de apenas 0.9 puntos
(95.5 % alineables) y con *stride* 4 de 3.9 puntos (92.5 %); con 6 ya se pierde demasiado (84.1 %). Por eso
se prueban 2 y 4 en el ajuste. Las secuencias no alineables no rompen el entrenamiento porque `CTCLoss` usa
`zero_infinity=True`.